# Minor Project - CNN for CIFAR10

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [2]:
# Datasets and DataLoaders

from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# Image => Scale (0, 1) => Normalize (-1, 1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = CIFAR10("./data", train=True, download=True, transform=transform)
testset = CIFAR10("./data", train=False, download=True, transform=transform)

In [3]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

## Build The CNN

In [4]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 1st 2 is kernel size, the other one is Stride
        
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
                
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10),
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # Flattening
        x = self.fc_layers(x)

        return x

In [5]:
model = CNN()

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## Training CNN

In [8]:
epochs = 10

for epoch in range(epochs):
    model.train()
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images) # FP
        loss = criterion(output, labels) # Loss function
        loss.backward()
        optimizer.step() # Update Parameters

        epoch_training_loss += loss.item()

    print(f"Epoch: {epoch+1} => Loss: {epoch_training_loss/len(trainloader)}")

Epoch: 1 => Loss: 1.3859406945193211
Epoch: 2 => Loss: 0.9493545116213582
Epoch: 3 => Loss: 0.7603532079692996
Epoch: 4 => Loss: 0.6315531899099764
Epoch: 5 => Loss: 0.5256674586773833
Epoch: 6 => Loss: 0.42766564261273043
Epoch: 7 => Loss: 0.34145242600794645
Epoch: 8 => Loss: 0.26853005320329193
Epoch: 9 => Loss: 0.2031670531729603
Epoch: 10 => Loss: 0.15566394348983723


## Evaluate CNN

In [9]:
correct_labels = 0
tot_labels = 0

model.eval()

with torch.no_grad():
    for images, labels, in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        tot_labels += labels.size(0)

print(f"Accuracy Score = {(correct_labels/tot_labels)*100}")

Accuracy Score = 75.35
